In [2]:
q1='I just discovered the course, can I still join?'
q2='I found out about the program just now, can I still enroll?'

In [3]:
from sentence_transformers import SentenceTransformer

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
print('q1:', q1)
print('q2:', q2)

q1: I just discovered the course, can I still join?
q2: I found out about the program just now, can I still enroll?


In [6]:
#encode strings to vectors using the model
v1=model.encode(q1)
v2=model.encode(q2)

In [7]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

print('similarity between q1 and q2:', model.similarity(v1, v2))
print('similarity between q1 and d:', model.similarity(v1, dv)) 
print('similarity between q2 and d:', model.similarity(v2, dv))

similarity between q1 and q2: tensor([[0.6196]])
similarity between q1 and d: tensor([[0.3957]])
similarity between q2 and d: tensor([[0.4311]])


### Embedding Previous Dataset

In [8]:
from ingest import load_faq_data

In [9]:
documents=load_faq_data()

In [10]:
texts=[]
for doc in documents:
    text=doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [11]:
len(texts)

1346

In [12]:
from tqdm import tqdm
batch_size=100
vectors=[]

for i in tqdm(range(0,len(texts), batch_size)):
    batch=texts[i:i+batch_size]
    batch_vectors=model.encode(batch)
    vectors.extend(batch_vectors)




  0%|          | 0/14 [00:00<?, ?it/s]

100%|██████████| 14/14 [01:33<00:00,  6.66s/it]


In [13]:
len(vectors)

1346

In [14]:
import numpy as np
X=np.array(vectors)

### Vector Search

In [15]:
query='Can I still join the course after the registration window?'
v_query=model.encode(query)

In [16]:
scores=X.dot(v_query)

In [17]:
scores

array([ 0.4104973 ,  0.27528238,  0.71485597, ..., -0.11804166,
        0.01468604, -0.09213875], shape=(1346,), dtype=float32)

In [18]:
#Take the highst score -> the most similar with the question
idx=np.argmax(scores)
idx, scores[idx]

(np.int64(536), np.float32(0.7433515))

In [19]:
documents[idx]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [20]:
#Retrieving Top 5 Results
top5=np.argsort(-scores)[:5]

In [21]:
for i in top5:
    print(scores[i], documents[i])

0.7433515 {'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}
0.71485597 {'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}
0.71052796 {'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks

In [22]:
from minsearch import VectorSearch

vindex=VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [28]:
question='What is this course about?'
v_question=model.encode(question)

results=vindex.search(v_question, num_results=5)

In [ ]:
results

[{'id': '2c26baf6bd',
  'course': 'stock-markets-analytics-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course focus on analytics, predictions or coding?',
  'answer': 'Both analytics and predictions, assuming you can already do the (relatively simple) coding - no OOP or complex data structures required. It targets intermediate-to-advanced programming/analytics skills and beginner-level trading skills.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '365688f181',
  'course': 'stock-markets-analytics-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course cover technical and fundamental analysis?',
  'answer': "Yes, on the practical side: it shows how to generate technical indicators from API data and a few fundamental features (e.g. earnings per share). The catch is tha

In [ ]:
#filtering by course
vindex.search(v_question, num_results=5, filter_dict={'course':'machine-learning-zoomcamp'})

[{'id': '742d3dbb55',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How long is the course?',
  'answer': 'Approximately 4 months, but it may take longer if you want to engage in extra activities such as an additional project or writing an article.'},
 {'id': 'bc2e78caf3',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What are the Slack “house rules”?',
  'answer': 'Slack house rules:\n- Ask course questions in `#course-ml-zoomcamp` (not `#general`).\n- Use threads to reply.\n- Paste text/code instead of screenshots or phone photos.\n- Don’t tag instructors; many peers can help and instructors see messages anyway.\n- Keep the channel tidy and on-topic.'},
 {'id': '36c30c8e2f',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How much time do I need for this course?',
  'answer': 'Around ~10 hours per week.\n\nYou c